In [ ]:
!pip install gurobipy

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import gurobipy as grb

Q1

In [ ]:
# Setup
I, J = 100, 5

np.random.seed(0)
U_j = np.array([1, 3, 2, 1, 2])
U_i_j = U_j[None, :] + np.random.normal(size = (100, 5))

# Dual optimization problem
m1 = grb.Model()
pi_i_j = m1.addMVar((I, J))

m1.setObjective( (pi_i_j * U_i_j).sum(), sense = grb.GRB.MAXIMIZE)
m1.addConstr(pi_i_j.sum(axis = 1) == 1)
m1.addConstr(pi_i_j >= 0)
m1.optimize()

# q_j
q_j1 = pi_i_j.X.sum(axis = 0) # as q_j = sum of pi_i_j across all i
print(' ')
print('q =',q_j1)

Restricted license - for non-production use only - expires 2026-11-23
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 600 rows, 500 columns and 1000 nonzeros
Model fingerprint: 0x8f90ea6c
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 600 rows and 500 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.4005136e+02   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.400513579e+02
 
q = [ 3. 58. 16.  4. 19.]


Q2

In [ ]:
# Setup
Pi_i_j = np.array([[0.5, 1.0, 0.938],
                   [0.939, 0.6, 0.829],
                   [0.976, 1.0, 0.732]])

I, J = Pi_i_j.shape

# Primal optimization model (to find p_i)
m2 = grb.Model()
x_i = m2.addMVar(shape = I)

m2.setObjective(x_i.sum(), sense = grb.GRB.MINIMIZE)
m2.addConstr(Pi_i_j @ x_i >= np.ones(J))
m2.optimize()

# Solving for p_i and V
x_optimal = x_i.X
V = 1 / x_optimal.sum()
p_i = V * x_optimal

# Dual optimization model (to find q_j)
m2_dual = grb.Model()
y_j = m2_dual.addMVar(shape = J)

m2_dual.setObjective(y_j.sum(), sense = grb.GRB.MAXIMIZE)
m2_dual.addConstr(Pi_i_j.T @ y_j <= np.ones(I))
m2_dual.optimize()

# Solving for q
y_optimal = y_j.X
q_j2 = V * y_optimal

print('p =', p_i.round(3))
print('q =', q_j2.round(3))
print('V =', V.round(3))


Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 3 rows, 3 columns and 9 nonzeros
Model fingerprint: 0x722ee3b3
Coefficient statistics:
  Matrix range     [5e-01, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.01s
Presolved: 3 rows, 3 columns, 9 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   3.000000e+00   0.000000e+00      0s
       3    1.2051704e+00   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.03 seconds (0.00 work units)
Optimal objective  1.205170446e+00
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count:

Q3

In [ ]:
# Setup
m3 = grb.Model()

alpha_x_y = np.array([
    [-16, -4, -8, -20],
    [-12, -8, -4, -4],
    [-4, -8, -8, -16]
])

gamma_x_y = np.array([
    [4, 8, 4, 2],
    [4, 3, 6, 6],
    [9, 4, 8, 2]
])

X, Y = alpha_x_y.shape

mu_x_y = m3.addMVar((X, Y))

n_x = np.ones(X)  # mass 1 of each worker type
m_y = np.ones(Y)  # mass 1 of each firm type

taus = [0.00, 0.25, 0.50, 0.75]
results = []

for tau in taus:
    phi_x_y = alpha_x_y + (1 - tau) * gamma_x_y # surplus for each tax rate

    m3.setObjective((mu_x_y * phi_x_y).sum(), sense = grb.GRB.MAXIMIZE)
    m3.addConstr(mu_x_y.sum(axis = 1) <= n_x)
    m3.addConstr(mu_x_y.sum(axis = 0) <= m_y)

    m3.optimize()

    A = (mu_x_y.X * alpha_x_y).sum()
    capital_gamma = (mu_x_y.X * gamma_x_y).sum()
    results.append((A, capital_gamma))

print(' ')
print('τ         A         Γ')
print('----------------------')
for tau, (A, capital_gamma) in zip(taus, results):
    print(f'{tau:.2f}   {A:.1f}   {capital_gamma:.1f}')

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 12 columns and 24 nonzeros
Model fingerprint: 0x8fde7cc8
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 7 rows and 12 columns
Presolve time: 0.02s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.1000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.100000000e+01
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logic

In [ ]:
# Final results

# Q1
print('Q1')
print('q =',q_j1)
print(' ')

# Q2
print('Q2')
print('p =', p_i.round(3))
print('q =', q_j2.round(3))
print('V =', V.round(3))
print('')

# Q3
print('Q3')
print('τ         A         Γ')
print('----------------------')
for tau, (A, capital_gamma) in zip(taus, results):
    print(f'{tau:.2f}   {A:.1f}   {capital_gamma:.1f}')

Q1
q = [ 3. 58. 16.  4. 19.]
 
Q2
p = [0.265 0.124 0.612]
q = [0.274 0.426 0.3  ]
V = 0.83

Q3
τ         A         Γ
----------------------
0.00   -12.0   23.0
0.25   -12.0   23.0
0.50   -8.0   17.0
0.75   0.0   0.0
